## 7. Kenya Application / Business Evaluation

The U.S. findings are specific to a U.S. reporting system. The value of the workflow is still relevant to Kenya, but the model itself cannot be treated as a Kenyan model without local data and local validation.

### What may transfer
The following parts of the project are transferable:
- CRISP-DM structure;
- time-aware validation;
- transparent feature and model evaluation;
- a disciplined approach to cleaning and investigation;
- the use of operationally relevant road and weather features.

### What does not automatically transfer
The learned relationships are not portable by default because Kenya differs from the U.S. in several important ways:
- road design and traffic mix;
- reporting practices and severity definitions;
- weather and seasonal variation;
- traffic exposure and trip volume;
- emergency-response coverage and reporting lag;
- local infrastructure and land-use patterns.

### Data available in this project
This project contains only U.S. accident records and variables. It does not include Kenyan road network data, traffic exposure, emergency-response outcomes, or local crash reporting metadata.

### Data required for a Kenyan deployment
A Kenyan implementation would need local records that match the U.S. concepts but reflect Kenyan conditions. The required data would cover accident timing, location, severity, road environment, exposure, and outcome.

| U.S. variable / concept | Meaning | Kenyan equivalent needed? | Reason |
| --- | --- | --- | --- |
| `Severity` | Severity score recorded in the U.S. dataset | Yes | Severity definitions and coding practice may differ |
| `Start_Time` | Time of accident | Yes | Needed for temporal and seasonal analysis |
| `State` / `County` / `City` | Geographic location | Yes | Kenyan administrative and road geography differ |
| `Weather_Condition` | Reported weather state | Yes | Local weather classification and reporting may differ |
| `Visibility(mi)` | Reported visibility | Yes | Local meteorological conditions and measurement practices differ |
| `Temperature(F)` | Ambient temperature | Yes | Climate conditions differ by location |
| `Wind_Speed(mph)` | Wind condition | Yes | Measurement and interpretation may differ |
| `Junction`, `Crossing`, `Traffic_Signal` | Road and traffic control context | Yes | Kenya has different road design and control layouts |
| `Sunrise_Sunset` | Light condition proxy | Yes | Local travel patterns and illumination differ |
| `Distance(mi)` | Distance-related context | Likely | May need local road-length or exposure measures |
| `high_severity` | Binary outcome derived from severity | Yes | Requires a local operational definition aligned with local policy |

### Additional Kenyan data required
A Kenyan system would need the following before it could be retrained and evaluated locally:
- geographic crash locations with reliable road identifiers;
- full incident records with severity and outcome information;
- weather and visibility data tied to the same time and place;
- road-network variables such as road class, speed limits, and traffic control type;
- traffic volume or exposure data to convert counts into risk estimates;
- vehicle type, road user type, and injury outcome data when available;
- emergency-response timing and fatality or injury outcome records;
- a local validation period for model retraining and performance review.

### Business recommendations
Finding: The U.S. data show that time, weather, visibility, and road context are associated with different high-severity proportions.  
Implication: These variables are operationally useful for monitoring and investigation.  
Possible action: Use the same variable categories as a starting point for a Kenyan data collection design and reporting template.  
Limitation: The U.S. patterns cannot be treated as local risk estimates without Kenyan validation.

Finding: The machine-learning task is feasible on the U.S. data when the target is defined as a binary high-severity flag.  
Implication: A Kenyan system could follow the same workflow but would need a locally defined target and a local training set.  
Possible action: Build a Kenyan prototype only after collecting a representative local dataset and evaluating it on a time-based holdout period.  
Limitation: Model performance and threshold choices from the U.S. data will not transfer automatically.


In [ ]:

error_df = test_df[['State', 'high_severity']].copy()
error_df['prediction'] = selected_pred
error_df['false_negative'] = ((error_df['high_severity'] == 1) & (error_df['prediction'] == 0)).astype(int)

state_error = error_df.groupby('State').agg(
    total=('high_severity', 'size'),
    actual_high=('high_severity', 'sum'),
    false_negatives=('false_negative', 'sum')
).query('total >= 200').copy()

state_error['false_negative_rate'] = state_error['false_negatives'] / state_error['actual_high'].replace(0, np.nan)
print('False-negative rate by state (states with at least 200 test records):')
print(state_error.sort_values('false_negative_rate', ascending=False).head(10))


In [ ]:

feature_names = selected_model.named_steps['preprocess'].get_feature_names_out()
model_step = selected_model.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model_step.feature_importances_
    }).sort_values('importance', ascending=False).head(15)
    print('Top features by importance:')
    print(importance_df)

    plt.figure(figsize=(9, 6))
    sns.barplot(data=importance_df, x='importance', y='feature', color='blue')
    plt.title(f'Top feature importances: {selected_model_name}')
    plt.show()
elif hasattr(model_step, 'coef_'):
    coefs = model_step.coef_[0]
    coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
    coef_df = coef_df.loc[coef_df['feature'].str.contains('State|Weather|day_of_week|Sunrise', case=False, na=False)]
    print(coef_df.sort_values('coefficient', key=lambda s: s.abs(), ascending=False).head(10))
else:
    print('This model does not provide feature-importance output in the standard scikit-learn form.')



## Final conclusion

### 1. Findings from the U.S. dataset
The U.S. dataset contains a usable severity label, timestamps, weather values, visibility measures, and road-context indicators. The observed patterns show that time, weather, visibility, and road context are associated with different high-severity rates in the U.S. data.

### 2. Model results
The model results should be read as a U.S.-trained prediction task. The selected model learned associations from the U.S. training data and was evaluated on a future U.S. holdout period. This is not a Kenyan validation.

### 3. Potential Kenyan application
The analytical workflow is potentially relevant to Kenya as a decision-support framework. The same CRISP-DM structure, time-aware validation, and dashboard logic could be adapted after a Kenyan dataset becomes available.

### 4. Limitations
This project does not claim that U.S. patterns are Kenyan patterns. It does not claim that the model is validated for Kenya. It does not estimate risk per trip or per road segment because exposure data are absent.

### 5. Data required for Kenyan validation
A Kenyan implementation would require local accident records, road and weather data, traffic exposure, and outcome/response information for retraining and validation. Without those data, the model should not be treated as deployment-ready for Kenya.

### Final statement
This notebook demonstrates a disciplined data-science workflow for accident analysis using U.S. data and frames that workflow for a Kenyan road-safety context without overstating what the data can support. The value lies in the method, the honest separation of evidence from assumptions, and the explicit requirement for Kenyan validation before any deployment decision.
